In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('')))

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.graph_objects as go
import plotly.subplots as sp
from datetime import datetime

# LUCiD imports
from lucid.geometry import generate_detector
from lucid.simulation import setup_event_simulator
from lucid.generate import read_photon_data_from_photonsim
from lucid.utils import spherical_to_cartesian, base_dir_path
from lucid.optimization.grid_search import get_detector_bounds
from lucid.utils import generate_random_event_params
from lucid.detector_params import ParticleParams, load_detector_params

## Configuration Parameters

Adjust these parameters to customize the visualization:

In [ ]:
# Configuration parameters
CONFIG = {
    'detector_config': base_dir_path() + 'config/SK_geom_config.json',  # Detector configuration file
    'physics_config': base_dir_path() + 'config/SK_physics_config.json',  # Physics configuration file
    'data_file': '../data/water/muon/muon_gun_1050_MeV_100_events_fixed_energy.root',
    'detector_type': 'Cylinder',  # Detector geometry type
    'entry_idx': 2,  # Which entry to use from ROOT file
    'n_photons': 15_000,  # Number of photons to simulate
    'K': 6,  # Number of scattering iterations
    'seed': 71900,  # Random seed
    'min_charge': 1.0,  # Minimum charge threshold for display
    'color_by': 'charge',  # Color sensor hits by 'charge' or 'time'
    'dark_theme': False,  # Use dark theme for disc visualizations
    'log_scale': False,  # Use log scale for disc visualizations
    'save_figures': True  # Save figures to files
}

print("Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Setup Detector and Simulators

In [ ]:
# Setup detector
print("Setting up detector...")
detector = generate_detector(CONFIG['detector_config'])
sensor_positions = jnp.array(detector.all_points)
detector_bounds = get_detector_bounds(detector)
n_sensors = len(sensor_positions)

print(f"  Type: {CONFIG['detector_type']}")
print(f"  Sensors: {n_sensors:,}")
print(f"  Bounds: {detector_bounds}")

In [ ]:
# Setup simulators
print("Setting up simulators...")

# Prediction simulator (regular physics simulation)
prediction_simulator = setup_event_simulator(
    json_filename=CONFIG['detector_config'],
    max_sensors_per_cell=4,
    n_photons=CONFIG['n_photons'],
    temperature=0.0,
    K=CONFIG['K'],
    detector_type=CONFIG['detector_type'],
    is_data=False,
    physics_config=CONFIG['physics_config'],
    default_detector_params=True,
    hit_mode='aggregated'
)

# Data simulator (transforms reference photons)
data_simulator = setup_event_simulator(
    json_filename=CONFIG['detector_config'],
    max_sensors_per_cell=4,
    n_photons=CONFIG['n_photons'],
    temperature=0.0,  # Zero temperature for data mode
    K=CONFIG['K'],
    detector_type=CONFIG['detector_type'],
    is_data=True,
    physics_config=CONFIG['physics_config'],
    default_detector_params=True
)

print("  Simulators ready")

## Load Reference Photons and Generate Track Parameters

In [ ]:
# Load photon data from ROOT file
print(f"Loading reference photons from ROOT file...")
photon_data = read_photon_data_from_photonsim(CONFIG['data_file'], CONFIG['entry_idx'])
photon_data['N'] = len(photon_data['photon_origins'])

print(f"  Number of photons: {photon_data['N']:,}")
print(f"  Primary energy: {photon_data['energy']:.1f} MeV")

# Generate track parameters
print("\nGenerating track parameters...")
key = jax.random.PRNGKey(CONFIG['seed'])
track_params = generate_random_event_params(key, detector_bounds)
track_energy = photon_data['energy']

track_position = jnp.array([-10., 0., 0.])
track_direction = jnp.array([1., 0., 0.])


original_direction = jnp.array([0.0, 0.0, 1.0])
true_direction_norm = track_direction / (jnp.linalg.norm(track_direction) + 1e-8)

# Rotation axis = cross product of original and target directions
rotation_axis = jnp.cross(original_direction, true_direction_norm)
axis_norm = jnp.linalg.norm(rotation_axis)

# Handle case where directions are parallel (axis_norm ~ 0)
rotation_axis = jnp.where(
    axis_norm < 1e-6,
    jnp.array([1.0, 0.0, 0.0]),  # Arbitrary axis when parallel
    rotation_axis / (axis_norm + 1e-8)
)

# Rotation angle = arccos of dot product
rotation_angle = jnp.arccos(jnp.clip(
    jnp.dot(original_direction, true_direction_norm), -1.0, 1.0
))

# Set rotation parameters
photon_data['rotation_axis'] = rotation_axis
photon_data['rotation_angle'] = rotation_angle
photon_data['apply_rotation'] = jnp.array(True)

# Set translation parameters to move from origin to true_position
photon_data['apply_translation'] = jnp.array(True)
photon_data['translation_vector'] = track_position


print(f"  Position: [{track_position[0]:.3f}, {track_position[1]:.3f}, {track_position[2]:.3f}] m")
print(f"  Direction: [{track_direction[0]:.3f}, {track_direction[1]:.3f}, {track_direction[2]:.3f}]")
print(f"  Energy: {track_energy:.1f} MeV")

## Simulate Events

In [ ]:
# Simulate events
print("Simulating events...")
event_key = jax.random.PRNGKey(CONFIG['seed'] + 1000)

# Create ParticleParams for simulation
track = ParticleParams.from_cartesian(
    energy=track_energy, position=track_position,
    direction=track_direction, t0=0.0
)

# Prediction-like event
print("  Generating prediction-like event...")
prediction_charges, prediction_times = prediction_simulator(track, event_key)

# Data-like event
print("  Generating data-like event...")
data_charges, data_times = data_simulator(track, event_key, photon_data)

print("  Events generated successfully")

## Event Analysis and Statistics

In [ ]:
# Create statistical comparison plots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 6))

# Filter active sensors
pred_active = prediction_charges > CONFIG['min_charge']
data_active = data_charges > CONFIG['min_charge']

pred_charges_active = prediction_charges[data_active & pred_active]
pred_times_active = prediction_times[data_active & pred_active]
data_charges_active = data_charges[data_active]
data_times_active = data_times[data_active]

# Charge distributions
ax1.hist(pred_charges_active, bins=150, range=[0,20], alpha=0.7, label='Prediction-like', color='blue', density=True)
ax1.hist(data_charges_active, bins=150, range=[0,20], alpha=0.7, label='Data-like', color='red', density=True)
ax1.set_xlabel('Charge')
ax1.set_ylabel('Density')
ax1.set_title('Charge Distribution Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Time distributions
ax2.hist(pred_times_active, bins=100, range=[0,200], alpha=0.7, label='Prediction-like', color='blue', density=False)
ax2.hist(data_times_active, bins=100, range=[0,200], alpha=0.7, label='Data-like', color='red', density=False)
ax2.set_xlabel('Time [ns]')
ax2.set_ylabel('Density')
ax2.set_title('Time Distribution Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Charge vs Time scatter
ax3.scatter(pred_charges_active, pred_times_active, alpha=0.6, s=10, label='Prediction-like', color='blue')
ax3.scatter(data_charges_active, data_times_active, alpha=0.6, s=10, label='Data-like', color='red')
ax3.set_xlabel('Charge')
ax3.set_ylabel('Time [ns]')
ax3.set_title('Charge vs Time Correlation')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Summary statistics
stats_text = f"""
Prediction-like Event:
  Active sensors: {len(pred_charges_active):,}
  Mean charge: {np.mean(pred_charges_active):.2f} ± {np.std(pred_charges_active):.2f}
  Mean time: {np.mean(pred_times_active):.1f} ± {np.std(pred_times_active):.1f} ns

Data-like Event:
  Active sensors: {len(data_charges_active):,}
  Mean charge: {np.mean(data_charges_active):.2f} ± {np.std(data_charges_active):.2f}
  Mean time: {np.mean(data_times_active):.1f} ± {np.std(data_times_active):.1f} ns

Track Parameters:
  Energy: {track_energy:.1f} MeV
  Position: [{track_position[0]:.2f}, {track_position[1]:.2f}, {track_position[2]:.2f}] m
  Direction: [{track_direction[0]:.3f}, {track_direction[1]:.3f}, {track_direction[2]:.3f}]
"""

ax4.hist2d(prediction_times[data_active & pred_active], data_times[data_active & pred_active], bins=(np.linspace(0,300,40), np.linspace(0,300,40)))
ax4.plot(range(0,300), range(0,300))
ax4.set_xlim(0,300)
ax4.set_ylim(0,300)
ax4.set_xlabel('Prediction Times [ns]')
ax4.set_ylabel('Data Times [ns]')
ax4.set_title('Time Alignment')

plt.tight_layout()

if CONFIG['save_figures']:
    figures_dir = Path(base_dir_path()) / 'figures'
    detector_name = Path(CONFIG['detector_config']).stem.replace('_geom_config', '')
    filename = figures_dir / f'{detector_name}_statistical_comparison.png'
    _ = plt.savefig(str(filename), dpi=300, bbox_inches='tight')
    print(f"Saved: {filename}")

# Lets study the Nray dependence

In [ ]:
from tqdm import tqdm

# Configuration
NRAYS_VALUES = [10_000, 25_000, 50_000, 150_000, 250_000, 500_000]
N_EVENTS = 30
DATA_FILE = '../data/water/muon/muon_gun_1050_MeV_100_events_fixed_energy.root'
DETECTOR_CONFIG = base_dir_path() + 'config/SK_geom_config.json'
K = 7
SEED = 42

print(f"nrays values: {NRAYS_VALUES}")
print(f"Number of events: {N_EVENTS}")
print(f"Data file: {DATA_FILE}")

In [ ]:
# Setup detector
print("Setting up detector...")
detector = generate_detector(DETECTOR_CONFIG)
sensor_positions = jnp.array(detector.all_points)
detector_bounds = get_detector_bounds(detector)
n_sensors = len(sensor_positions)

print(f"  Sensors: {n_sensors:,}")
print(f"  Bounds: {detector_bounds}")

In [ ]:
def prepare_photon_data(photon_data, track_position, track_direction):
    """
    Prepare photon data with rotation and translation for a given track.
    Returns a copy with transforms applied.
    """
    # Make a copy to avoid modifying original
    pd = dict(photon_data)
    
    # Compute rotation from original direction (0,0,1) to track_direction
    original_direction = jnp.array([0.0, 0.0, 1.0])
    true_direction_norm = track_direction / (jnp.linalg.norm(track_direction) + 1e-8)
    
    rotation_axis = jnp.cross(original_direction, true_direction_norm)
    axis_norm = jnp.linalg.norm(rotation_axis)
    
    rotation_axis = jnp.where(
        axis_norm < 1e-6,
        jnp.array([1.0, 0.0, 0.0]),
        rotation_axis / (axis_norm + 1e-8)
    )
    
    rotation_angle = jnp.arccos(jnp.clip(
        jnp.dot(original_direction, true_direction_norm), -1.0, 1.0
    ))
    
    # Set transforms
    pd['rotation_axis'] = rotation_axis
    pd['rotation_angle'] = rotation_angle
    pd['apply_rotation'] = jnp.array(True)
    pd['apply_translation'] = jnp.array(True)
    pd['translation_vector'] = track_position
    
    return pd


def compute_metrics(data_charges, data_times, pred_charges, pred_times):
    """
    Compute the 3 metrics on sensors where both data and pred have hits.
    """
    active = (data_charges > 0) & (pred_charges > 0)
    
    d_times = data_times[active]
    p_times = pred_times[active]
    d_charges = data_charges[active]
    p_charges = pred_charges[active]
    
    cov_metric = float(np.mean((d_times - p_times) * (d_charges - p_charges)))
    time_residual = float(np.mean(d_times - p_times))
    charge_residual = float(np.mean(d_charges - p_charges))
    
    return cov_metric, time_residual, charge_residual


print("Helper functions defined.")

In [ ]:
# Load photon data from ROOT file (use entry 0 as base)
print(f"Loading reference photons from: {DATA_FILE}")
base_photon_data = read_photon_data_from_photonsim(DATA_FILE, 0)

# Pad to 1M photons (required by simulator)
N = len(base_photon_data['photon_origins'])
padding_size = max(0, 1_000_000 - N)

base_photon_data['photon_origins'] = jnp.pad(
    base_photon_data['photon_origins'], ((0, padding_size), (0, 0)),
    mode='constant', constant_values=0
)

default_direction = jnp.array([0.0, 0.0, 1.0])
padding_directions = jnp.tile(default_direction, (padding_size, 1))
if padding_size > 0:
    base_photon_data['photon_directions'] = jnp.concatenate(
        [base_photon_data['photon_directions'], padding_directions], axis=0
    )

base_photon_data['photon_times'] = jnp.pad(
    base_photon_data['photon_times'], (0, padding_size),
    mode='constant', constant_values=0
)
base_photon_data['N'] = N

track_energy = base_photon_data['energy']
print(f"  Photons: {N:,}")
print(f"  Energy: {track_energy:.1f} MeV")

In [ ]:
# Results storage
results = {nrays: {'cov': [], 'time': [], 'charge': []} for nrays in NRAYS_VALUES}

# Master random key
master_key = jax.random.PRNGKey(SEED)

for nrays in NRAYS_VALUES:
    print(f"\n{'='*60}")
    print(f"Setting up simulators for nrays = {nrays:,}")
    print(f"{'='*60}")
    
    # Setup simulators ONCE per nrays value (avoids JIT recompilation)
    prediction_simulator = setup_event_simulator(
        json_filename=DETECTOR_CONFIG,
        max_sensors_per_cell=4,
        n_photons=nrays,
        temperature=0.0,
        K=K,
        is_data=False,
        physics_config=CONFIG['physics_config'],
        default_detector_params=True,
        hit_mode='aggregated'
    )
    
    data_simulator = setup_event_simulator(
        json_filename=DETECTOR_CONFIG,
        max_sensors_per_cell=4,
        n_photons=nrays,
        temperature=0.0,
        K=K,
        is_data=True,
        physics_config=CONFIG['physics_config'],
        default_detector_params=True
    )
    
    print(f"Simulators ready. Processing {N_EVENTS} events...")
    
    for event_idx in tqdm(range(N_EVENTS), desc=f"nrays={nrays//1000}k"):
        # Generate random event parameters
        master_key, event_key = jax.random.split(master_key)
        track_params = generate_random_event_params(event_key, detector_bounds)
        track_position = track_params.position
        track_direction = track_params.direction
        
        # Prepare photon data with transforms
        photon_data = prepare_photon_data(base_photon_data, track_position, track_direction)
        
        # Create ParticleParams for simulation
        master_key, sim_key = jax.random.split(master_key)
        
        track = ParticleParams.from_cartesian(
            energy=track_energy, position=track_position,
            direction=track_direction, t0=0.0
        )
        
        pred_charges, pred_times = prediction_simulator(track, sim_key)
        
        data_charges, data_times = data_simulator(track, sim_key, photon_data)
        
        # Compute metrics
        cov, time_res, charge_res = compute_metrics(data_charges, data_times, pred_charges, pred_times)
        
        results[nrays]['cov'].append(cov)
        results[nrays]['time'].append(time_res)
        results[nrays]['charge'].append(charge_res)
    
    print(f"  Cov metric:      mean={np.mean(results[nrays]['cov']):.3f}, std={np.std(results[nrays]['cov']):.3f}")
    print(f"  Time residual:   mean={np.mean(results[nrays]['time']):.3f}, std={np.std(results[nrays]['time']):.3f}")
    print(f"  Charge residual: mean={np.mean(results[nrays]['charge']):.3f}, std={np.std(results[nrays]['charge']):.3f}")

print("\nAll simulations complete!")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

nrays_labels = [f"{n//1000}k" for n in NRAYS_VALUES]

# Covariance metric
cov_data = [results[n]['cov'] for n in NRAYS_VALUES]
bp1 = axes[0].boxplot(cov_data, labels=nrays_labels, patch_artist=True)
axes[0].set_xlabel('nrays')
axes[0].set_ylabel('Covariance-like metric')
axes[0].set_title('mean((data_t - pred_t) * (data_q - pred_q))')
axes[0].axhline(0, color='r', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3)
for patch in bp1['boxes']:
    patch.set_facecolor('lightblue')

# Time residual
time_data = [results[n]['time'] for n in NRAYS_VALUES]
bp2 = axes[1].boxplot(time_data, labels=nrays_labels, patch_artist=True)
axes[1].set_xlabel('nrays')
axes[1].set_ylabel('Time residual (ns)')
axes[1].set_title('mean(data_times - pred_times)')
axes[1].axhline(0, color='r', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)
for patch in bp2['boxes']:
    patch.set_facecolor('lightgreen')

# Charge residual
charge_data = [results[n]['charge'] for n in NRAYS_VALUES]
bp3 = axes[2].boxplot(charge_data, labels=nrays_labels, patch_artist=True)
axes[2].set_xlabel('nrays')
axes[2].set_ylabel('Charge residual')
axes[2].set_title('mean(data_charges - pred_charges)')
axes[2].axhline(0, color='r', linestyle='--', alpha=0.5)
axes[2].grid(True, alpha=0.3)
for patch in bp3['boxes']:
    patch.set_facecolor('lightyellow')

plt.suptitle(f'Data vs Prediction Comparison ({N_EVENTS} events, E={track_energy:.0f} MeV)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print("\nSummary Statistics")
print("="*70)
print(f"{'nrays':<10} {'Cov (mean +/- std)':<25} {'Time (mean +/- std)':<25} {'Charge (mean +/- std)':<25}")
print("-"*70)

for nrays in NRAYS_VALUES:
    cov_mean = np.mean(results[nrays]['cov'])
    cov_std = np.std(results[nrays]['cov'])
    time_mean = np.mean(results[nrays]['time'])
    time_std = np.std(results[nrays]['time'])
    charge_mean = np.mean(results[nrays]['charge'])
    charge_std = np.std(results[nrays]['charge'])
    
    print(f"{nrays//1000}k{'':<7} {cov_mean:>8.3f} +/- {cov_std:<8.3f}   {time_mean:>8.3f} +/- {time_std:<8.3f}   {charge_mean:>8.3f} +/- {charge_std:<8.3f}")